<a href="https://colab.research.google.com/github/jaylink-coder/miss-yaya/blob/main/yaya-ai/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Yaya-125M Training — Google Colab

Trains the Yaya-125M model (~129M parameters) on OpenWebText.

**Requirements:** Runtime → Change runtime type → **T4 GPU**

**Estimated time:** ~10–12 hours for 100k steps (full Colab session)

## 1. Check GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('PyTorch:', torch.__version__)

## 2. Mount Google Drive (for persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/yaya-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CHECKPOINT_DIR)

## 3. Clone Yaya repo

In [ ]:
REPO_URL = 'https://github.com/jaylink-coder/miss-yaya.git'

!git clone {REPO_URL} /content/miss-yaya
%cd /content/miss-yaya

fatal: destination path '/content/miss-yaya' already exists and is not an empty directory.
/content/miss-yaya


## 4. Install dependencies

In [ ]:
!pip install -q sentencepiece datasets wandb
print('Dependencies installed.')

import os, sys
sys.path.insert(0, '/content/miss-yaya')
os.chdir('/content/miss-yaya')

from datasets import load_dataset
import numpy as np

# 2% sample (~16k docs, ~13M tokens) — tokenizes in ~2 minutes
print('Downloading OpenWebText sample...')
ds = load_dataset('openwebtext', split='train[:2%]')
print(f'Loaded {len(ds):,} documents')

In [ ]:
from src.tokenizer.tokenizer import YayaTokenizer

TOKENIZER_PATH = 'data/tokenizer/yaya_tokenizer.model'
tokenizer = YayaTokenizer(TOKENIZER_PATH)
print('Tokenizer vocab size:', tokenizer.vocab_size)

TRAIN_DIR = 'data/processed/openwebtext/train'
EVAL_DIR  = 'data/processed/openwebtext/eval'
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

split = ds.train_test_split(test_size=0.01, seed=42)
train_ds = split['train']
eval_ds  = split['test']

def tokenize_and_save(dataset, out_path, filename):
    all_tokens = []
    for i, row in enumerate(dataset):
        tokens = tokenizer.encode(row['text'])
        all_tokens.extend(tokens)
        if (i + 1) % 5000 == 0:
            print(f'  {i+1:,} docs, {len(all_tokens):,} tokens')
    arr = np.array(all_tokens, dtype=np.uint16)
    arr.tofile(os.path.join(out_path, filename))
    print(f'Saved {len(arr):,} tokens to {out_path}/{filename}')
    return len(arr)

print('Tokenizing train split...')
n_train = tokenize_and_save(train_ds, TRAIN_DIR, 'shard_00000.bin')

print('Tokenizing eval split...')
n_eval = tokenize_and_save(eval_ds, EVAL_DIR, 'eval.bin')

print(f'\nTotal: {n_train:,} train tokens, {n_eval:,} eval tokens')

In [ ]:
from src.tokenizer.tokenizer import YayaTokenizer

TOKENIZER_PATH = 'data/tokenizer/yaya_tokenizer.model'
tokenizer = YayaTokenizer(TOKENIZER_PATH)
print('Tokenizer vocab size:', tokenizer.vocab_size)

# Tokenize and write shards
TRAIN_DIR = 'data/processed/openwebtext/train'
EVAL_DIR  = 'data/processed/openwebtext/eval'
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

# Split: 99% train, 1% eval
split = ds.train_test_split(test_size=0.01, seed=42)
train_ds = split['train']
eval_ds  = split['test']

def tokenize_and_save(dataset, out_path, filename):
    all_tokens = []
    for i, row in enumerate(dataset):
        tokens = tokenizer.encode(row['text'])
        all_tokens.extend(tokens)
        if (i + 1) % 10000 == 0:
            print(f'  {i+1:,} docs, {len(all_tokens):,} tokens')
    arr = np.array(all_tokens, dtype=np.uint16)
    arr.tofile(os.path.join(out_path, filename))
    print(f'Saved {len(arr):,} tokens to {out_path}/{filename}')
    return len(arr)

print('Tokenizing train split...')
n_train = tokenize_and_save(train_ds, TRAIN_DIR, 'shard_00000.bin')

print('Tokenizing eval split...')
n_eval = tokenize_and_save(eval_ds, EVAL_DIR, 'eval.bin')

print(f'\nTotal: {n_train:,} train tokens, {n_eval:,} eval tokens')

Tokenizer vocab size: 32768
Tokenizing train split...
  10,000 docs, 12,760,891 tokens
  20,000 docs, 25,249,986 tokens
  30,000 docs, 38,046,816 tokens
  40,000 docs, 50,819,522 tokens
  50,000 docs, 62,836,032 tokens
  60,000 docs, 75,452,904 tokens
  70,000 docs, 88,157,974 tokens
  80,000 docs, 101,073,016 tokens


## 6. Update config to save checkpoints to Drive

In [ ]:
import yaml

with open('configs/training/train_125m.yaml') as f:
    train_cfg = yaml.safe_load(f)

# Point checkpoints to Drive so they survive session resets
train_cfg['checkpointing']['save_dir'] = CHECKPOINT_DIR

with open('configs/training/train_125m.yaml', 'w') as f:
    yaml.dump(train_cfg, f)

print('Checkpoint dir set to:', CHECKPOINT_DIR)

## 7. (Optional) Login to W&B for live loss curves
Skip this cell if you don't want to track metrics.

In [ ]:
import wandb
wandb.login()  # paste your API key when prompted
print('W&B ready.')

## 8. Train!

In [ ]:
# Check for existing checkpoint to resume from
import glob
ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))
resume_flag = f'--resume {ckpts[-1]}' if ckpts else ''
print('Resuming from:', ckpts[-1] if ckpts else 'scratch')

!python scripts/train.py \
    --model_config configs/model/yaya_125m.yaml \
    --train_config configs/training/train_125m.yaml \
    {resume_flag}

## 9. Quick generation test
Run after training (or after loading a checkpoint).

In [ ]:
import torch, glob
from src.utils.config import load_model_config
from src.model.yaya_model import YayaForCausalLM
from src.tokenizer.tokenizer import YayaTokenizer
from src.inference.generator import TextGenerator
from src.training.checkpointing import CheckpointManager

model_config = load_model_config('configs/model/yaya_125m.yaml')
model = YayaForCausalLM(model_config)

# Load latest checkpoint
ckpt_mgr = CheckpointManager(save_dir=CHECKPOINT_DIR)
ckpt_mgr.load(model)

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

tokenizer = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
generator = TextGenerator(model, tokenizer, device=device)

prompt = 'The history of artificial intelligence'
output = generator.generate(prompt, max_new_tokens=100, temperature=0.8, top_p=0.9)
print(output)